# ⚡ Google Drive GPU Refiner - Remote Worker Node (Helper)

This lightweight helper notebook provides external GPU compute (NVIDIA Tesla T4 / V100 / A100 / L4) to accelerate video refining for the **Main Google Drive Downloader & GPU Refiner**.

---

### 🌟 How it Works:
1. **Shared Drive Integration**: Mounts the shared `downloader` Google Drive folder.
2. **Zero Configuration**: Automatically discovers the Main coordinator via `data/cluster/master.json` on Google Drive.
3. **Hardware NVENC Acceleration**: Receives 3–5 minute video chunks, transcodes them at maximum GPU speed using NVENC hardware acceleration, and saves the finished segments directly back into Google Drive.
4. **Fault Tolerant**: If this Colab runtime stops or is preempted, the Main coordinator automatically reclaims the chunk and reassigns it to another worker without losing any completed work.


In [ ]:
#@title 🛠️ Step 1: Environment Setup & Google Drive Mount
#@markdown Mounts Google Drive, verifies NVIDIA GPU hardware acceleration, and prepares workspace.

import os
import sys
import shutil
import subprocess
import json
import base64
import zlib

print("=" * 72)
print("🚀 [1/3] Checking Hardware Acceleration (NVIDIA GPU)...")
print("=" * 72)
has_gpu = False
try:
    gpu_info = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
        text=True
    ).strip()
    print(f"✅ Active GPU Detected: {gpu_info}")
    has_gpu = True
except Exception:
    print("⚠️ No NVIDIA GPU detected! Helper will run in CPU fallback mode (slower).")
    print("👉 Recommendation: Click Runtime > Change runtime type > Select T4 GPU.")

print("\n" + "=" * 72)
print("📁 [2/3] Mounting Google Drive...")
print("=" * 72)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("✅ Google Drive mounted successfully at /content/drive")
except Exception as e:
    print(f"⚠️ Drive mount notice: {e}")

print("\n" + "=" * 72)
print("📦 [3/3] Deploying Helper Modules...")
print("=" * 72)
workspace_dir = "/content/colab-gpu-helper"
os.makedirs(workspace_dir, exist_ok=True)
os.chdir(workspace_dir)
if workspace_dir not in sys.path:
    sys.path.insert(0, workspace_dir)

# Decompress and write modular files
bundle_b64 = "eJztfQtb40aW6F+p0DeDnRgD/cgkzDK7NDjdTGjoxXRn5wOuPtmWQUGWPHpAe1j++z2PKqmqVJJtOrN7d79xvjS2VM9Tp867Tj1ujJN4Gt7054uNPbERzuZJmosku4rl1zQovxZFOCl/5OGsepPdFnkYVe9u08CfhPHNVTxNk5nIF3P4IeTbs3keJrEf9cRJmOU9cRSO4d+DeCFLz/38NgpHqvhH+HkVX8UvxFGQB+NcDOL7ME3iWRDne+JdktxEgThMIn8k7jNxkox9GMjx0Ds8Ozl4K/ZhLn1ssR98gd6yztXGNsw4h8pXG13hxxMRJ3m90OHeFX42utj1JJgCHLIkug+8SfIQR4k/CVJvEqadrtj6s8jydO8qFvC52qD/6PvRIvZnIQwnWogIhpUHmRruURreB6JqSkBTMLUkXfREVsxx3giw7Ba+jIs863GLffVgK/fTmyDPtkaLrXDCYysQqj2aUXbrp8FETJMI2s7Ew20Ifd4UfurDxANseQpPMm4UiorTwefBuXhIwxwAIwcDo84Tc8BpkuT9+kTDKcFQwVyCAj8jPwsQTNoy+KMM/3bUb3gLYAo6nodj8rxut6qeBnmRxmXV35Iw7qgmezAEbS2y3L9BhOMFw8ovxG5ffEzDmZ8uxPg2GN/ticMiTWHlxUOS3iEYSrCLDkxilhQAn4koYlwSfeZyTOOHiXsmsBbwrqPGDm3Bzz6MKc2zhxDKVFi3PcEGtxXyYZMIvDAWl3bDdiWo06v1Xmv5w+Jo9cJDwhT6nkGNa23t8nSh/cIPVEtDQOJ9cRn0o+QhQPSfJqkIcPjQVwSbBzcFTKp7bdYFkPjxonOHJVU7WJUeXMJS+rl/tYGLmvoP8kswDeNgcrVx3RVQEqsH/SCeKIj2w/kiHiEgyzHIlrvWwDVcgqHVBmZtfgPdoDxhWjm8cVRkeZDyj5mP3/u/ZUkM41i12+DLOJgDHaM/sGmtanM/yyokftkXh4i8jZu/Y2xRVUjIQiKJmThK1JSPPdzKsJA1hGjoRdvqFrT0Brut6IOrlMOATVxpql+242Fv2rajZdFr9bDVbr2mJEuqXphRd9ScqyP8ICzCuAgcw6hNp1zNVXeF7Pra3cwSPOTKX4WK6iNRkltsHMx/725dY6xZMaphRwWskkMQZ1gN8NAiE2cTdfDx8qFCqXqZJTteferotxapeKVIxdiPkxglD/w2CWG1gpLVhYrvl68Idat2mxgKPEUZa2KILrD2q9bcomdbz6jp0TNvvZr6yq9XXmyJoSSC7VV15vkM6NSrrwUio7o+W64jyQxutjFutmq5NRyq4/7YRnE34QP5PtJ2Hf4kAWPspsNUXN9Rsjd87tpq9Lxhk2hjVmINN7OqGLWWhGQJRXU40KbHAViy0vIdz5u3YWoMh9pqaH2Oq43/ui+GgZ+Ob4V7poQFQJGlRkB0xg9jkoCBkm9LRoJ0e+bn41t8gUI5tz9bMFY7pIUSkk3SgazbLhhUHFTjlqomTO8tjC4TP38aDjSBfeyDTF8XMLCthcZcHPCbFlHkzW2eIfvrcQurihPc1u8pTiyRA7jD1eUAXNN12/tbERRBfzJaTY7gFp7N8+q7AD+IfR5SGFoNJVa5F4WKmsqWwfYRBPCMKOzVBuKGbB3pTlVWf9U07eZlY4C4iaJEkuaKvzOF1Jr9/4dSVkBqoJjqsyLe4MeNO1ovNm6uJU69Kakqs1rWsGSHmcZ9XZTRVKybyKPeyjNopFH9d6SFerv/2wjiPwnaGlMGZaaOHRrE27Urp2YFyOHWrZqxptKsnr2Xj4KpX0S5mPpRNPINFanSLKWJd9Ii6TSI+YYdtEbBuVV9dqv0VBfptVp1ui47KUdSFv5HEHpnIYsBrD9HNUvJILg6Pj46+/X05OzgaHDuHR2fQ4NNxvirGGZu2aFPzg4PTrxfz85/GX48OByYw8mD2Ry7DqIskOXfHgwHspt1bdb1royNo1omEqV8EPiDR9FlL8cwT4sxgAAgZ1j2jnQF/vzgV2uM1IMJqNI+Ay2fD34+Ph0crVhJ2XK0ih/OPh8Phq76WtNEb5P7kIzJVd2LzyvUy++9LEjtqmcX78tVb6ud5LeSCh0dXBysNk1mGFDlw+DigKr9ZXh2Wqsn26O5BbmPtRQ34RUbzG+DWZACLSHvk/j8wVwsRouGJbNwRq6ZF3LzsirNddXauHjKFXJxfjB87w0vDt4dn75bsYE89bNbw6HChZoA62iihCy/a4Wv2bgbyheDDx9l11ZvsN7nx58VbMw1ll5DoMJpOCrQs/Pu4ydxKJXdP4jD2yK+y2DD0UzNNWsBWR2RdFgdvv90+suwDeTaK6IENAoNWrKF1SCt1T48+TS8cG8WHbqlcKRVwSYH584+tVaxNnrOgtTs8MMB/fn5+KRG8qzapixWTrh54HXk0IdvVm+ZRK0feypmgZYJOVuyp2WjnUK5Q3K1F6mP8opEFe/o0/nBxfHZqTccHEJvL1/v9HdKKeaFeC1mIDGjlXYOLdwFi2kKrEdkwQ06wEUnAeFnBkRn5Ed+PA5EMhUs6uRJBDQJH/1BgLgpuQhZUY63zxSmeicD5EoXxx8GZ58u5Bhe7cgxgBJUdh8FfgYyjxy96MzT4B5GkKFMBS/8EUh17JXPsBTuqNdbM7Q8IoqCgO/H2TiZkKbCS+W9HxycX7wdHFxY/e++ge5fiA9+eid4lWBa0wh4EomOkyAK8oDFLXELKlo+CvycVJ7dN7B3Pxz8h3dweIFkQS3UX87eDhG4FVwPWRVJCgBsOA+w8T0Bc9oaFdMpes3j4EsufktGmxnPIBNZQgRE4g0UuIeBZSHo0pMo4FX/HKSjBKBxktwQTfkDSJ6j4kacJnk4BXETQZeJC3iLNY4Gbz+9807O3g33xChJUD1AGSPgAAf0KoNcNTg9eHsCBK4syzLDLioC36Aws0M4N/wrTPQDFiC83cOghDYykC0AJWegiNxUKAs0F93rFHKR7Znih4qtKH3n42S+UNT33z8NPsEQ37Z1qFmV1H5rqlbf9UZlHOuxdGv2hB89+ItMFDDuiAY4HB6J4b+fhIAiKJ5hFAMsL2AiukGMOdFmwDJbUJMmNfXDCESv7E+CBIqHEFrFls1pusRMbS7m5AzhUitlN8lhJsUcbfS2YBsHD/iXFpUiTk6TWDXJkRh6sMkcYAibEL4rpzBwNf8mIFUnQ3j4gLwPUHISwpYELAYa5QpG6etRHjdRAjRG2NxPijXwRRfI6lJj9Yylwep3KeKZ/ZSrb0gPPVHy+54wOGmd8cKzik4bjTtYRlVYo/7Qhbmtetay9YS+gJoSKFcMVfW6kUS+NDVC1HpUIzWFx9ZEVAtcfG1tgCo9VyPQKz9HK9DrP0Mz0Kuvrx0QdNfXELDaV2gJ1GuLnIrvny1xYuWvkTqp/vqyo17tefKj3sLzZUiCnblL1+B71cqszcO03e4IfWul9FgAxRW2zlSVnk1cqyaWUtmeRlrXIaA912pLX3Or6RqgOfPvAiBXWWfSE2QW85K7/Yu0UIF1+FnRngfqdpwBiwYKOw6yDIP7tNAGbv0qtoH7lYD9rwVqz2lN6gnDjlD+LG0DPVHT9lWhanC2ilk+0cfUqFddxWqpzWVeYYlblrdaWhDKxxSiOV6IYTDz57cJCGMgRf/KWsAJiHG5lPDVkqrxod7Cb07OfvWGHwe4qvUyOyj87wBqTEIOyf10enL84fhicITeibEMEAWFaSubB5pcJHuV4K7aa1A2SJ6TVYF7yalA+TI2ul9OsOOaDoANBuHRIDxnSygD4mRIFhRFjDIuaptjDYi4DzKcDCiHIAmn4fguCvQ5QWGQ/zJYkJVGaU5fyuIX4SxIipx0syEB7VeMIpgkN+K8oDjjcnKAm+e2tvcTKJtakWr1Lt6fD4bvz06OvLcfEdS7OzviO/j35es+rSL+/uXtduaubGqUL6kTA9A8lvPBxTlsdlYOUa8I8xAkxO/hJ8hk0uf2CoTmHB20OZpq80zpIFuzYIbBCBd+dicuUl8qEQB09G9uYYy8iFACnyXQbpJSSHwOhSnm6lp+91D7MIAOmtYdWrPxtTemsGRU5na433csxErNGC3JaRJlGr7xG88f5xy3gdvwKmYZbqWXjQNiJSXzEUp/D8j8TZZw/KfSTux4+KEsnwlVgeOS0iTLtuaRn8OvGUxkBlw3HIVRmC8ImczIVn8a5KZKomRt7NwWpaHr+3ASJKos9Yqeg35WjDrp5iWG929/9697Vxv/8uf/vN7s0Vh7VK7rrnJ1lX1P5URZsI9Gnrly5Mme8c3lHqDcNY4QJkzg6Yo/IxoK1AZldA1Dc7RAAuix2ZVtGx3yK6AM4sHGJJ9JBV20FDiOG7ylZuwzBhfYqngbKnr7IUGHNYjDrBvTgQujAqiOCb2PlC07KM3biB2gf4yxcprlkk9B4RiNUzMgMTEeG0hhU2AvYc6NRIu+uLgNM1GajPSx/a2ArYVYnBUz4gzNJwzqULH1K+lzqxes61riZ7RbLeNmDpbaJMAUcQjyoSexZqoM6Y94bKaP/7zudPu3wZfLvR+vn7xHNWS0uNAecgz6SQEAPwRLQJAst0VUxyC1wWhDfEHrj8Y0c9ETXHRecDoZQjJyVY2P9/RxVR2j7GkjM/r6BCwpvmMc48YnYXZXlaj7h7WG6kIko1XH2ZlcUqZlLnFD+CAb6jIH0OK8A4t0SVjIm+caGFeKvpM9kESKiE9m8PYUtJhEth7rIHjaE4/Bk+EEr8vB9em2o2l96o7y9eI1ULjA4RSurRbk/mAiNY4CPy7mMhQXw+49oEf+glgACH3l473yfNclFLiWckrXQbCGYz/O6geMRMftLObAcepUgAqGO5+UQOMs03gMNB/YJprB1HEm1P0DWMbqbER1XkqEWVYAqYLGyBh4c5vA9honc+D7krzRgSrgPRphgxZnosiQ1zuIt5t+VRCCPll2U4fPHNqjMgfh9yr+UmsDBrxCNKaDNDoiMk3bk4m4rbGZSyV7s+UXaFgGSSbOtKWcKKMzQkNZTvQ1ZTSbq5pGrDqwZpQaoHAnvdr4v6yef3d1BfjR/+7/IJeGEsfvTs/OB4cHw0G311i1tNlgbUSrtvrXWmAJjhb4Whak9+wYGQVTlKHlhtlTBY+nld1pkgBEcW1ufVhXZVSg8FvddIS9Z+MiIHY8Qrpf8WxEAtd6IV30JrUTEa1GjGZkka3ZVIm2f72TMoS2oQ/Zjx0fw42ZUUF4uAcExurln8WOmzSWnLqaYhODVh/JztCBMaX15056CniOKhW7OOR1vRbntDYTce9H4aRaRFodjZoBY32U7T5ppxPLtcIVX9+YuGTJsJJz0Vy9acvW3FnL2lFnzavHr//h64fd9CqQPmcNjdk3LyQWq3j96oJG2asUMvhQaqUw0lYnwldJE9wO0RKKpNqpbIct0Z0coefXj801BrnXYjkJsGU7Ni7V4zhxRFASx2MS7LoQBG/6dMygw6NwHY1wLRW6EPBQNLENWhI/zcMpKLIIL2zqqYay+HHqV+XM6uUVtL/fF7uO1kAlvisZrCprordj+IeyYDEXj7LSk+KAo+1KjnFoZHzqWjk+z4uYJy8ZDB7wnCWTAorLA/MY/FfEhGGh9JK6HJZLJDroDtjfRpTc3ARpY3YA0BTcGQHQi2k8aM8JYGUC6JVCpLQwUeUtFMNAk7gpHZp/UNaPq9iD53W7xbkyXMDCj4obDzStURRMSnc7UhjkzgN+jlLiRIWK4h7zM3wANdHb0sOoUWDEQArQfz9BPQpaToMxYB/2DxsXJ3KJ80Dpl+ZyfS1tPR7bJt9574+HF2fnf+VACznyUrR3y8+l5SXIeSYdYyameM2TQXEC9C36ei/DEqgqQpCCE3wywJCfmDEoczl8TcjxK4xKFiXENcw3C8PgcXRqrNpeq6IcLp0RDtcUprmLIcxTu1EyoFDYQ6u2xYjF2S1q1AszXlT9QWeuWeJnRa8Erk2YyaVxW2nOSRbOGOVgVu41Ad1AjsFYC6mS2aOskELhUMeBQyZ2qFegz2R4QNxH200GaIt2bxIwiXMg+iNKi07Qv+kjlTESIjgxxUDkZZhiFIYV4Do8Iw8Ngn7uIQXJcn82t/NsSHgoKtMvv8TJQ4esclP82bna+Pb93rcf9r4daqk8PCR9qYcmpwT6x9CfjgmjX8M5kGI8MTcPYiQk4uoKD6+P0eDK5AtEcuBvGFUSCYpMInoKQJW76mwolGXT3FYmqnpI0zxUVXGw0Aug/PdsYhTfiZ/e0A96vLofheaotZsHX3I2H1pWWVJqqQxFhYBylxRAgP0iTwD6MpokDWAeY4RBEQfxOJkQRfkUhxjMBTKMn1KSgSi48ccLIYHaNmXgGX3uql+Nr+t8P42K7LZjzrwjux7E9G+aJkBnz4b0ZdkZmjHgGQinIJqknaoblHxpZuRR77HBgjXsIp9u/agTGhogghYHjawG/vSpcgBkbgySFY4j28eQCYQbRsv3J4F6b4kbNVCUTTcXNGCyJomKgvsg8rLFbARUmX64jPbRPbJGet0v5vPqoApas/Ed0ubj05/PrjZcZver4uXuq5+uimmwM1WgCyK9LlHepsqTH19N4N/JzqRWGcQZ2CS/Hpyfcgwcfjs+hZYc1l0axw/+Tts4BufnZ+eNk/jj67G72vDT4eFgOGyuuPPGotv0eOflSz7lQPszmIVEtbVl6Incv5HfZujbvgnkL6AiYyCUKFaWoguZ51x2tUM0Pyh+gt1guAS/OpgjPcuk7gBbPixdWBz22NPIAQpaxA9CYjGcWojGQK/lTndYuxppvjLjV7OhNgkCBmaUpjFLbLM0WsO6hJ8c7UMO5mHasxZQ6NFs6mqjLA3LKsqMS9V7GiS+oy+117ByVNG/qb2SS4mv5dd6bZSUsTqyMGJc2pCfNG6hyZp9nxaz41Dl0OWkF0X125Y+LVgaLc+TeWfHMFXAvDwOKEWt5hF+Pl2zcAZflUSm00leg5xkQK6SPV2LxzoB6j6JR9k6fJMAeqqOA+HnBalPKbCZm0QxY+K50tZGqESH1oMHQltyM2lgczF8Nw+uxs2MN7ZsNiiOmvK6CUf3OUbdzlE7RGQ0111u/qDtlcxxjY2qSBHpAKTiZfsl/+qiVWLacLxu2m+bul5y5aQGTONiDHpedNxEzaRc78Ob2y1YxiRF32+sRUuLGaqjOdrLOzLUl5Y7wLRnN346gbcZRr2zHM2qmSmeVrRWsS2itOV4DPqqoC5F62KMUUYrzWHIZVccffMQS/ay8ihZ7VhljEcg/ISoRE1C/yZOMhDyaMN0FEwTFGFDBc1KG2keriTZbYNlVqVG++Cn8UqDVXYy2s0mWFcGphIWVoQkSW8rDY7kza8ampQ+Vh4bCK46le5QvM8eOuw5NoakOJf9wamGcktMLaWPB2hEVh0+nEAjphjfyNellIPuIYPvXG7xGK+b5kCirnsidgyJGndFpWZFlIfEBjAIA/AkmTpmtdoMKjeXc/w1EcZ9or6U9k5ZeoqNUx/QcJJOYOCLAO15VVWcgwoIKomylSTLZjMmPw4uN+HJ5rViyuq3izOXPSoBomxBSUDYjsWq4TX93rw2+TU8l4gLlQx+UUm+yEjY/EzddnWXcsqobG4tYvaZJpsiLG/DrHYkodm4oEsz1I+yZ94GESg0Ms5pfbMm+UWc+VCtFKhFGkXhqJ8GfysCDEg1HxOZsR/OYdJaStViNE8TZCgrmlAdxlPOsHpVjVhZw+RPNu7qe/NmXnhhPE3Y3mLSEWONjuNsHgCBEBwrIm6BEz9gKlFE3NPPx0fHB3RYCkuZexCbNwXwqw3slkKiYDlg/aEeB3xaGbDuU3/mzUZYasd4QTbslGurfp5ctgcMU9zXYNsnH4wHj+dF3jE3yeXVRnwfAqPcymYhK5xbW7Cc6WILxruP4+0xgvYpOrDHo/BkPKWqwQRrf5zd9+LkNsATNvCliDGQdePaUgWQHu6boRtWjBl+5pgZAgnGXL1j/wslHCzyfjaPQjw61sMj+TXNgGqDSrAvXtk2U1iaS301rskuB8Uvd+x8nly0XBEsiU4PLr17bbsOubhap6rdl9crmrU0moKNaUbXGHjw2PPvQbBBUaXB/EqZ8TKEwennwemhuA3ux9u3L394zeIyHuhD9JXZd0kSgg0HApAkFc32rHacgllPp7N5cCPxQfWGa18tdw+Nb0AV9rVmjgafTz+dnLhoKg6e581JPnAIZK/CCVkvlsLXDvcZRxjv9J4IJW9Dh5HhoAChNpnh8Unc5ucUPpqqOO0YTYOVEyoL4acvkzQzBUbeGMAi3Un7wZE8iZaJDz6M+z70VRbjI/ZYwqjCWYb+szwRL/VYbQrzlGc05XZiIGfUTEmZaN3ZipFKYQIjQNAfOFHnaTMj+bE+AoedAzXqw7PTw0/n54PTCxlUj4HESmEkQw/QUxD5vE4WRNOexCYvnLj9PDDPBIQEPKuYpB4whvZwKvxgu31Ft9m+qZFxrRxGP+EbRFa9jrnj+9JuCeSDyTjjLZoxqzcXQRb51QtFhAgFTwHyurxBfZXThs6r71B+qo4EeY/lAFV2nKeG2Mla64huWtwlo67oPJrT3FSzBFGlW2uDjxW5QW6tiZpvP+UvnPuE3MFmQc7w7K7Mohn79oyByG1juCitIhy77WVRkmd7zKWB9lZ831YAcA6PO3sSw3b5y1NLo80B4EaVKemSHm89DyYaYQi6HBFJItMo8XPq3u6OwzgxnN5KEE6SojLSs8CCeGbnfSDks/K4dHtEG0sJzxBIdRuMYwR1u0u1h9UxWa9CEmqCOI2BLNq+1CkFfgZfbv0iQxiT5w0Ti8msukTxNDTZzIR2yE2MAn+McSXkV9I62O0rT6brEB2jHr1uTG5QtYVZpptzo/e39TSR29rgOO4HWDnlIui3FKz6wjS1xAnM8P5ipPLVNwSAbn+3SuuYC1Olza+11JDderWWMR8cMwSZ9s1u3cjIulKTP6Oun+mn0SuoozbN7JOik9i4AExJBtFzAszJ9izMMsz1T8EjKtl3vxkNyzywZDNUEQuS0F1em2bX3b7MWSHzhvMJclNLxX0q00K7UGyj2rzO13bOwLnmolO7H39It5wlVk7F3GGEtOaoFN1515wdID2nNEiDyKfTMNb0GtppSgtFlGvVPHTdr+2m3+d2V+5OnzrswV9ioNVWwli+p4KsFWFs3K1haBF16qxii7BvwzGlPlWUkRXCW+FpUzRzb4XCZrrFVWosSXSvPpbaQ+dKOYg71ufkQMKG8MymPPeECF+Vfl4L0OZezNS5joS5+qxWSBapPs6sgBNOxd+ULFKDyhqZ/0rAtO8QTlqJDa2+IeodrpXfUd9MR8HWpACVG8+TqlhwskemmNhOE3uCmGTvXBek5EEdMxu7ZQgcz40M3pJym8OL8QhdPVHE2M47S2gJRWUOWByUY6L4uO9PJh0s64BVbdRqSWR5c/ySeUB3tXq1YAWUJCyJUHnKuJmlQclVG877JBp3krZ/tCb4UKhkwUbimIYN9ew0oU13a6w3i8o7yLUpe8czvIL4oaMY+2T07OPB0s60YWejrrNPxWXSokpQbrt3QalIIH104HvXzHl6m+dzM7TD/lDIlcedl20YSllz3RfirEwhFN7E6EbeffnH/g78tyu2+bwaHQbyMzr3A2uvEj8B9uLuAdVofKvL7eq0Yyv1W4V3ElBgBuWA2JhTTZcU7XKI1tulxLSRy1Swkb5MDJM2sgTxukqV5BsxXMTj2zSJw78HZVSlI7WJ68OHLpz7fNlvuS1cNFwDdNmBK6u2fLcMVO35vjWA4szdqZvKrtrbWSPrtfo0Z7/Gj7olQGHF70SoZLMqJhkfoYaMO6KIx4GXJxxQLs1dsKHzIlPJyK42KFOasaelq5ejjsZJBCQYxrI1TQO14ZQCTLFIDvOcrkb2bXVHkrBauE9phEKngWmiqkXnlOalsmz5pFZW92S0WNlq9TTXhl2tsrHXajF4qRJ9q0ctwX71yhR5VohRrXQa3GAcMADVqxWuymqWHHXJVE2ir1M5VzaRjZ4pQyoS4so4ZCYPssfeZC9ao9eWVJNW31XXlpyGsgOSYgmWtUODcqcdyq7xQCE/tvzNdXVzqoHUT+4jXdReKTFwyzTbZ0gMJCZMitm8g5sOxtLDLCsgR+2/bCB+03ow63KAaYCbZsB/OtAKZpVIOk0M4atuFFgj5gm/k2Ob/D/AdrwUY3/SwFPkMW0ni5bjCj82PfscpOE0pDPIfq47qDazkgibFJNZHpqGnPSybPl4igkbHmCB+YzUREb+jxaGjEMMhe1SMoVITEk9jaO1W2MQjnJOARHOZoDd8CtaSN/LlqIzGeMextJrXbQYr/5JbyozTOah6Q9XYl9PZYGf/7m0yG004V4aL4TSIWF6S/SPdnBPfdbc23KAVX9WeXlWToY9Xm28H5x8HND6T43A9tfiwN6puO3UJOAn701xHtA+woebbaDc7Pf7NXCyU6culxHp2ec/dV3a9oW5TAPopJI72MMNLN0iNhl3JqKwnc34rJ5fgqHCPpYaMTwvszDoxE8nefUzlET9ihxl85qI+E+i8tWE4x9ABtYiAVWKFFnuq3m4h9YHT4ZvSa4dxBNKrStjQ1HK2Wv0uSpPveEidAdmVqg9pNMZfizeX1x8FB/Phheo6r8bXAg5EmaYozR5yIB4cDQRZ/S9T0JMNJ4Uk2mE0Q4jTF0j0xrXUF6S2fbNbqh66iEbW2BNH63KT48KPIZ/vo4vMBNPamVm0/hRM6ppbAyjTzDrrYMbujgDY70+JH8Po8jfxnzhnV9DzICWidMLsbvT3/mTgAc/vP6T+PLD6y6efImCX4PRL2G+/ebVH/uvfhCdX95ffDjpiSi8C8Q7ENmSrji8BdIRbFO6OPxPDP2pn4aySoNn4WCMeMUj8udsCIb13mb0dlb5jy2OlNk6PqoroGaFpxqNJuiFnEFESwTdAORSLM9ILu+q02qVUF+vLpcBXXrsNNm6WMwDdSy3PsfaCkM5MwKyfy63EjzmrbOvxthT3e3Lv9aACOWt1uAnaSzwu0daalLk+692umyty1xOQb4TByPmkPPhcaOXO66cFPgZJZMFZX6D8hgB0SnP8LVBjedOm6Y0mWYdbKt+Wk8PBO3jbud4cit5hBw3UnE/nHmYDvBqQ5nuS3JUH4mURCjS3pJDiLI8YvKbSfCEfLHat5hGAacLY69nUnBSA9dkPp2fPH8u8gorXNEJBa+VJdC+G3RVaNB6cwYkjoNxle/pd5q2zHtJs12Fdi5LFVJC6Cumz0dBrPlXPdYmvuKMG8x+TjGwjIxxqrLnle7H1r3Sokf7vIz+Izw9+HhMGMFC3FsS8WrMDAbhzf0F7rb/NZY+p7VtNf1iHOwKBWTUHCqw6mp8p8a/u7YSkZIvkoqZ0tDVxrY/D8vQFoUEnC6rXAvzSGJK5+lR+8+ki0jZLrt88rOyPRrnavEDoll4w/lvqurqoYRzz1ouC6kl6NRxLgt2hrejGkgD6GDXbj6q7p82vzGApt31YPVdI0xXG3zjEKM6dysvV5FZB/4k0MCF+YlgQ5jRUmzKGakdYbhc2zW/0tpU292sqVW7O4NRVMZjKQTPMCnuOFtN8HVSABZyKS3uPE1uUjwzh6hR3eQC8ixCvbbTq10uR4HSsR5ZKAtcGrv+WmGxQ8BaFcc1E3pP1BGcUnLKxxZ694QD4NRhq5lQael2pRaF3txw7tHIzVZlO3DjqpOLVBTmp1fivQLIHl+jJV486j1uUkwogPrLZvcJiI3xbj7ON3sCpOvuXn93+vRtV/ybMEtM5xmWgKrwzYWqlFZmKU1J7rSdwVYGJXo44jeXKWcfkyiS4emoJ9P9QOWpA3nLEZMFjCSvIW8p4pD6ZGBXNax/LXF039aW/1DStv1H/cRQHxPosr9NI39PJodswHI1pmW0+tbPpMBmix6/cUx3VfS3ZES7DpfPLDpGdDDKlkiCxbd2rfIvBJ3ZoEPmlalHHvEDQs3Rx/wu+OIDIaX2rE5lULJXxHkYKSi4A5dpVB2cUY8H22UsrUlomltM/IvVRbPFjC7QYuTT+3Apbw4JTNudSwWAN+IQMQpvHpN7U8Fc25e4QOJ7sfu0Xb2lc0wSLLhxk6n4SzLS6/Py0qbe1OqFeRRAi5tQGt5tdpukyjRY7ktWhzKqfdoT3C2dEe2Jcg7yNwjKPGaZQItTZuCxeMxCCALVHsehY/68/k4LYypGeL2A3MpyGMiOEeDjYAW29FXCp9o5e3K2tQLadtmrgFArZoIDy5pPahV0UGFx47dLEF2Vaxor6eacqzGc4LW8QQ1QuiaPJVO5Yi8eS6A86cvXYRxmqD51bSl3DYbik6VH3oOkq0Ha3l6KsM0IKOeYGdMDrJPtC58ZGN7xQBOmm4wxJFwmqZKmDU4nmVuHZf/7MfWrcEmH8XNQ6eWuL3MSgXZETVXUsRwh0cOK6kmMseH8FQgkZ2EjUHlzA4+EErk5bc5slPf46EpJ7Zox6kjd9CFS/0FijRlHQ76S2p4qNXBoPfBnyPSk4gEKCdoeK1A837JseS0bTcsqpG9zG3iL28y83OSobIyPthWZ7EMgw737+IkPjm3bktfVxlNrYveVDZSv36xhoPymzUDphqAxIC0GVMMX8gKNZCQH8oWmaA4ryW4y+g2xe87c1o4DlcNxhq9qnbuzBDuK/VklusXPKmaztr1PeMyoL7FZbTdO9wHqCdk/Ws1hNYclW9F5w8q9iqRpzzo038Prd3LFClACWSXKY/AlGNOlq+rw6hZyHrzPFeMy+ABzeaEqZd8rb77EBStZokaYM8kjAMFD0P3Glbqd9fgKD856L/D26OwWuunP7u6NeDs7YqRmiPMfpP+Wob1PELnkC611iUQzbyE2OevYUsy1vtmn3nh6I8tKQquuvfZkjgNY+0d92zItr9pXPExrt2IDZSmNkRlHo/i5PHJau8nDAoTBoxDJkRZbvlLniURA3jCmE5mPEouevMeqaxNZuTIfRl+paYTwkrar1lGmmFICvnIKvapLWyt0XHWxdiy+uvPi9zypsdsXPwf5+Ja4oTzyvScu0oVpXaPbbUSHyCgJUtsU628cxQPdcOpHkRINJpopj6mMDrsinlRX1jjDdeTNUghh2Nc3Qef1cpjaaObOwW4Xcmdir43RHUjjCKJ5wacY5ZlO6xAjww32JvCOWwwlA1KmDjbKU6NlYbNZN27Yh4VVSHjzjmsG37wtAlw/DOII3145osCYEpoNsigI5p3dvplfD0ZnrsBKYRe1RPgmFHoVuVkyfBdLxU+LT+1yCFRDlLTjWm4dYj/SMtMBftr9E20TuipFbhVtj7jilvDTtGNqcqZV0H0bVAUFLGBvEO0tiOj23lhicnn5avqqSY1QWlklOPfFkKZdHa9eKoC7IMT9BHOXJRFfbSvFZVtTiG0n/QQI953SvFz6h+qlGYvkGlBLX4k8eq5WMUwKJCg1pZ7Bil0ughzv+gmyLKRM57EK80oZaFLF07GOL+howLcWo+BlR+q4lXjQRY+CbgP8XvD1gs5mTcNfra1G859Dtp96M/3SFQcGb4sO3o8or0l0O8EacfmPO2+acLnziH2TuV58eNslIywAnD1Tdd5XIfU3lhxBp67fFmE0ET//jGlxNDl2nMxmwMI0mRXmRZcM8vWHUu5TmrX2kt0scRGZx83G6bRWDZ55wMWxxis9Lv3ewzGMa+Upuwy/417e91/+8AYA/X7w+dC8jyXzKJ8Q6vnwl09lqWbpxNYB1tCeVnXRwE5pezCNkiOlUT0EFbuqSwhlO479eO+BosVhjhtb4717no6eMYgSFNEB1px/zF/Lh+O/cXkMQwAIdq1j0Q6vZ3OPoCZ/Ifhwy+lUb7g2hPsgXUx9POSm9eno73mT1/Io/VdO/s1zJq/h9WzC7RqZpTCdJ7AP+WthAOyFkL7mz+cHH6qUSBTUhFQSt/Hp56PB4R7musas9qh9YiK2ssgoyfMoiIOxeYNfE9BhjMB/czwTjAC4fSANVp7LLiY+DK9rzEgvTenWSrKmF8QQZZ0qfLNfbvz2AdwzpLW6jf0bzXzH61gzcW759dg+fIpxmF4c3FBCCS/P1OHzu8D7e5AmrjrKCy7RLpwHe7u1cqWuVT2+NsjCpEi9jAgYOTs6mm4sX2H7L1+jC6SeDeLEL2LQiyRNlinI2JBEnvrKOS+vsq3ZXRvZyg/+Lsg9eFdPrFP7ZNrEbFAfMg2wnc3HcjrS2YReJgpnak7yJDqmmFGLcMFpmqnbPpLVDLChJ29D0BOyfTz+OGhJ1GakcxsVU+TN+7sGqPk43ohSk5syhIaKRZp687H0VlnPp/PM+Zzvqsa9sLV1ZeTOrmsOdLMHpk0FRoRTkBcLuCQ4LLVPf+pJCNUHswpyAe3sNoxzv/nsdvvBLm2ijMncvMxpiM1e7l6r8bQcq5UmHIogoCxesl13ja86JEYNRE5I0Mq0wMJYvraZNoCSJ0kNaNPk36uOES1BiIZe8fxVw5BaygDpmkPLKvFwiW5BE9ti19vZ2cH/XQK1UYn3CPCqzk8/9Xd6YuZ/6ezgl07Z5jY93MWHkgZ2uyQbSyLY1L6EKvSgwRR//U7IptVYceXw89Uoip84eWiiPPoH8KSDRbcMmkVZQzG4vi07DNJjK3TMEcyvf5Z6FOsVVvKF16vRiu6V67m0PK3nXrmaS8vL1dzT1nJ5nfIQtx6Y1VzrqQVxl/hAkRXXdO8m84WUSvbEowIXh2thtJYCyF5/Z0pBWuI/xRBnq0rT1N338ZVjNZkhYFsLwWL3qRrTPqBXTeBTH4047DaTEXuLO4oSc3zww7ztth2X6Ywq3oVRZO8uuTycZX6ZTaTqAmgXyk4NC6Vwxu2/+geYNtY3a7icaPjBOEWEFb8nsRAE+p1WOx66G5rtePS22Y7nhDzf9dME3kpilWbNNYVOcQnDx9TbkwBWyJouJmz/n7Rg6iYWNSI6WMnR05rZpiXEDq+WcfReZvfTu3qN2QfnCy1LL5sEdZ+k7rep6rbdt2J5HFsuXLHN6yWG9Wy3pekqk6FTDkudXa3ZXtcesN5urDMsagQ5tN+pWLjShreu8iSGPlr8Kkewbdl70+corUXNnG3hhhnhV8OGGnh7JUytHn/o88Wqmv8b3WhJ6qcL/Q5xWpa1PJhuYXdFv6ZEwQYP5jNOnTcE6MtrDMiL60UJbC2ZXkILOXDk2uE4Ws7ty85HgaZy2EaUsR+XvQNys3hJ3zHWJ8U33XqwzySCNQwxr/q+HrHxcIshRkY2Y6fnYPlhc81l9sohSlSprKzVA7xSvg0t5tsRyYshG451bphYOUG0i7izJjeF09iFL+UiXctggwbvQcsxjkpWdHk05GVEEghGtAp2VyLJvvzblBjm95ulHuf7dd2gzdGPF55MQLKPN3V2Mv1ILKF1xgkOrYb7935U4AXHzeknzPZb57rCKZvGbuQaNanbGurXgtDx02B3turu9N/Y9MddT8N5983X6HfQ9gWUQTFMe/QtiB4u2Ut92pSTmvR7DO3uiV9B+lY3pFdnLfSM/00uPYtyVEQTKFJDYgniWvKUQUARWCGFViVz0FXQPIojMa4eQHopKzBu2RSy2TZKthHg+z/8WOf6SwL8gx93VAZt6F9mu8e0+2xmJSXMPA5iRwq19ALvJYM4PlLt6KGQa7QDo9trFzHsl/IAJL778JZ8Fd21euTIuPfKvXGgguaQy4rHzb8G2aZ0n9UvDaF8/Junieig2+NnGdvT3VxvzocKOcYLNXfnNRFPEn2B8+I9Vn4cJEUWLRSvXavPI7qr7SS5kUHyMoUzzpiv1j4Snc9823SX5i8b1C6s5skfHQ9l8Q/q2rqMboRbHQoGWuuiGqGmuXv0nVNOvDJck3ziSodLEkoZKNUM45pMZ9xowHfKy+jTfQ5At+WpnkDfz34HH/S6mLcARLrYlQwpZ4uqLWLISajsuLntyG2XgJaTgzH8O91FclCFjgzUQTNO3Y6XuSlU1I77uvXOprw+mmwHFPlZMh9F36lrXgQdUeCosFrsy6fzk2eKi1aJMrzFcX+Dm/sv72K9dTmqUqfSNLV1WmVNVlobu3CLXKAtnpvHSzDoiyz5vPboW/FDG5vXIOQ48yxjpnTGXkMAmcvJnbIOdZPqLoUm9r98qfTzRa9/EhfhfA9z4OHl51sKYRYgmt9hqikQJvDICfRL2epcKIvKVkJvN60rXzYx1BFYHvqrod4wD+biZfOwNcnFFt3Up0nveSE+1sK7+WZ3Hy+mJ2kZg0TZ1S3IIMTnhYDdCUmXgQsp1YJps9nHeuK6LIMqB5HynHNMtUvmpJnp4rx9+djUaNcZzoohtdjLkoL4WdNjoXsr/Nw69mrn9DIrGl6Lsq72tL269F6UFek3n1FtrSe9GGU9+r1CvdKbUdaUT3rS0byk+hqODdup0UDE3Etl9MR6l92eHnvLobdSF/DkfUEzn7KoWZdvqQSdnA4VJUV5IzneLGlc0WTdDku3KTAbRlyUQjpL7dXVZ5q2oAx+QS6Fslp3ckZSato3bmizR75v/ZZ1TauWvNkOFSL5XpqmfgkWowSE6OMYSFZazPXYgJUIajDZVVOWykSomsKsFQtRZNCGGN4WOXEBMg1blNxxzorH26JzJ9MpOotqpFXWdCYZbPHt2G4daZXDO8BJh/E8dkt5hD2eV7qkbLyCPjae/h+IKeyx"
try:
    raw_json = zlib.decompress(base64.b64decode(bundle_b64.encode('ascii'))).decode('utf-8')
    files_dict = json.loads(raw_json)
    for fname, content in files_dict.items():
        fpath = os.path.join(workspace_dir, fname)
        with open(fpath, "w", encoding="utf-8") as f:
            f.write(content)
    print(f"✅ Deployed {len(files_dict)} helper modules into {workspace_dir}")
except Exception as e:
    print(f"⚠️ Notice deploying bundle: {e}")

print("🎉 Setup complete! Helper workspace ready.")


In [ ]:
#@title 🚀 Step 2: Start Autonomous GPU Worker Loop
#@markdown Launches the helper node. It auto-discovers the Main coordinator and begins encoding video chunks.
coordinator_url = "" #@param {type:"string"}
#@markdown *Optional*: If Google Drive auto-discovery takes time, paste the Cloudflare or Gradio link from the Main coordinator here (e.g. `https://xyz.trycloudflare.com`). Leave blank for auto-discovery.
enable_debug_logs = True #@param {type:"boolean"}

import time
import logger
logger.set_debug(enable_debug_logs)
import helper_worker

print("=" * 72)
print("🟢 Launching Autonomous GPU Worker Node...")
print(f"   Coordinator URL: {coordinator_url if coordinator_url else 'Auto-Discover (via Google Drive master.json)'}")
print(f"   Debug Log Notifications: {'ENABLED (Verbose)' if enable_debug_logs else 'DISABLED (Quiet)'}")
print("   Keep this cell running to provide GPU compute.")
print("=" * 72 + "\n")

try:
    helper_worker.run_helper_main(coordinator_url=coordinator_url, enable_debug_logs=enable_debug_logs)
except KeyboardInterrupt:
    print("\n🛑 Worker stopped by user.")
